In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 0.85,
  'nonbondedmethod': 'LJPME',
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_T': None,
  'anneal_steps': None,
  'anneal_interval': 100,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 50,
  'dispcorr': False},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 24.188},
  'Lc_A': {'weight': 1.0, 'gt': 20.108},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}},
 'validation': {}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="jsdivergence")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", "NonbondedForce/epsilon", "NonbondedForce/sigma"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1097, 0.1467, 0.1153, 0.1538, 0.1719, 0.1453, 0.1612], dtype=float64),
  'k': Array([314553.12, 247609.12, 746007.2 , 194556.  , 237065.44, 571534.4 ,
         335473.12], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([1.87762521, 1.90956473, 1.91637152, 1.94970731, 3.11680898,
         2.08881005, 1.86785137, 1.69183491, 2.09526777, 1.85947379],      dtype=float64),
  'k': Array([ 326.352   ,  409.1952  ,  391.6224  ,  554.7984  ,  612.5376  ,
          530.5312  , 1026.7536  ,  567.551232, 1072.7776  , 1089.5136  ],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([0.        , 0.        , 0.        , 3.14159265, 3.14159265,
         4.68141978, 4.82988662], dtype=float64),
  'proper_k': Array([ 0.50208   ,  0.65084444,  0.65084444,  0.        ,  0.        ,
         27.18068704, 37.90498239], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dtype=flo

In [6]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[10].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[11].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list)

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble


Anisotropic pressure control


== Energy minimization ==


== Start Relaxation ==


#"Step","Potential Energy (kJ/mole)","Temperature (K)","Density (g/mL)","Speed (ns/day)","Time Remaining"


...

 20%|██        | 20/100 [00:00<00:00, 81.98it/s]

 43%|████▎     | 43/100 [00:00<00:00, 135.16it/s]

 66%|██████▌   | 66/100 [00:00<00:00, 165.90it/s]

 90%|█████████ | 90/100 [00:00<00:00, 189.97it/s]

100%|██████████| 100/100 [00:00<00:00, 168.91it/s]

In [9]:
trainer.fit(10, 2)

Pair list:   0%|          | 0/100 [00:00<?, ?it/s]

Pair list: 100%|██████████| 100/100 [00:00<00:00, 1103.53it/s]

Init Energy calc: Memory Usage: 1554.50 MB


Get weight, Memory Usage: 7658.92 MB


Finish loss calc Memory Usage: 7721.06 MB


...

grad obtained.... Memory Usage: 8071.88 MB
grad modify.... Memory Usage: 8071.88 MB
update finished.... Memory Usage: 8071.88 MB
Effective sample sizes:
  sample_0: 76
  Total: 77.89285531236987


Epoch 8, Loss: 100.51787714243734, Memory Usage: 8072.20 MB
Loss:  100.51787714243734
Best Loss:  100.51787714243734 at epoch  8
Epoch 8 completed in 5.40 seconds.
----
Init Energy calc: Memory Usage: 8072.20 MB


Energy calc: 2.10 sec. Memory Usage: 8299.86 MB


Get weight, Memory Usage: 8309.12 MB
Finish loss calc Memory Usage: 8309.55 MB


grad obtained.... Memory Usage: 8090.06 MB
grad modify.... Memory Usage: 8090.06 MB
update finished.... Memory Usage: 8090.06 MB
Effective sample sizes:
  sample_0: 63
  Total: 64.19157708550112
Loss:  99.19063605508921
Best Loss:  99.19063605508921 at epoch  9
Epoch 9 completed in 5.04 seconds.
----


In [10]:
trainer.get_loss_gradients()

Init Energy calc: Memory Usage: 8090.06 MB


Energy calc: 2.14 sec. Memory Usage: 8309.67 MB


Get weight, Memory Usage: 8309.88 MB
Finish loss calc Memory Usage: 8309.88 MB


(Array(97.98640548, dtype=float64),
 {'HarmonicAngleForce': {'angle': Array([ 303.07948178,   61.87361191, -182.387456  , -309.46372491,
           -71.17342579,   16.93863408,  111.38673627, -185.32192255,
           -86.2059518 ,  232.7899784 ], dtype=float64),
   'k': Array([ 0.02553812, -0.00647951,  0.00210702,  0.01125608, -0.00972446,
           0.00283565, -0.00736807,  0.00969045, -0.00095159, -0.0130538 ],      dtype=float64)},
  'HarmonicBondForce': {'k': Array([-1.62478175e-07, -3.84915689e-05,  3.15908389e-05, -3.58414145e-05,
           7.18997356e-05, -2.79917232e-05, -1.86030365e-05], dtype=float64),
   'length': Array([  -563.80931306,     33.76666879,  -9387.77755295,  -4071.92062762,
            1526.12503331, -11853.94653971,  -8075.52847584], dtype=float64)},
  'NonbondedForce': {'charge': Array([   0.5942975 ,   -7.98581244,  -42.51018827,  165.58936694,
            55.0019623 ,   56.81497881,   37.16638787,   78.4589951 ,
           283.11424572,  183.02986668,  

In [11]:
# Shortened for the example (the production run used fit(490, 2)).
trainer.fit(40, 2)

Init Energy calc: Memory Usage: 8090.91 MB


Energy calc: 1.98 sec. Memory Usage: 8310.52 MB


Get weight, Memory Usage: 8311.80 MB
Finish loss calc Memory Usage: 8311.83 MB


grad obtained.... Memory Usage: 8092.23 MB
grad modify.... Memory Usage: 8092.23 MB
update finished.... Memory Usage: 8092.23 MB
Effective sample sizes:
  sample_0: 49
  Total: 50.362357205316854
  0 -> Resample
Resampling sample_0... by xmlfiles/epoch_2rdf_1adf_density-11.xml
Using anisonpt ensemble


Anisotropic pressure control


...

grad obtained.... Memory Usage: 7958.97 MB
grad modify.... Memory Usage: 7958.97 MB
update finished.... Memory Usage: 7958.97 MB
Effective sample sizes:
  sample_0: 97
  Total: 98.55798597795858


Epoch 48, Loss: 57.28316916993091, Memory Usage: 7961.56 MB
Loss:  57.28316916993091
Best Loss:  57.28316916993091 at epoch  48
Epoch 48 completed in 5.35 seconds.
----
Init Energy calc: Memory Usage: 7961.56 MB


Energy calc: 2.16 sec. Memory Usage: 8430.09 MB


Get weight, Memory Usage: 8433.72 MB
Finish loss calc Memory Usage: 8434.08 MB


grad obtained.... Memory Usage: 7987.48 MB
grad modify.... Memory Usage: 7987.48 MB
update finished.... Memory Usage: 7987.48 MB
Effective sample sizes:
  sample_0: 93
  Total: 94.4768316434941
Loss:  56.61030719677797
Best Loss:  56.61030719677797 at epoch  49
Epoch 49 completed in 5.22 seconds.
----
